# Hito 3 - Notebook 09: Modelado Avanzado - Clasificacion Clinica
## Fase 4 de CRISP-DM (avanzada) - 1.5.2

> **Este notebook esta disenado para ejecutarse en GOOGLE COLAB** (GPU/CPU) y generar los artefactos `.joblib`. Compara **Random Forest Classifier** vs **XGBoost Classifier** para predecir `Prioridad_Atencion` (Bajo/Medio/Alto) en la **cohorte pediatrico-juvenil** (`Age < 25`, definida en el notebook 05).

**Metodologia:** el preprocesamiento (escalado, codificacion) y el balanceo (**SMOTE**) se encapsulan en un `Pipeline` de `imbalanced-learn`, aplicados de forma correcta en cada pliegue de validacion cruzada.

In [ ]:
# === Configuracion para Google Colab (ejecutar primero) ===
# Instala dependencias si no estan presentes.
try:
    import xgboost, imblearn, sklearn, joblib  # noqa
except Exception:
    !pip -q install xgboost imbalanced-learn scikit-learn joblib
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
sns.set_theme(style='whitegrid'); plt.rcParams['figure.figsize'] = (9, 5)
pd.set_option('display.max_columns', None)

In [ ]:
def cargar_procesado(nombre):
    """Carga un CSV procesado buscando en rutas locales o pidiendo subirlo en Colab."""
    for p in [Path('data/processed') / nombre, Path('../data/processed') / nombre,
              Path('/content') / nombre, Path(nombre)]:
        if p.exists():
            print('Cargado desde:', p)
            return pd.read_csv(p)
    try:
        from google.colab import files
        print(f'Sube el archivo: {nombre}')
        subido = files.upload()
        return pd.read_csv(list(subido.keys())[0])
    except Exception as e:
        raise FileNotFoundError(f'No se encontro {nombre}: {e}')

MODELS_DIR = Path('models'); MODELS_DIR.mkdir(exist_ok=True, parents=True)

## Carga del dataset preparado (desde la BD/CSV integrado)

In [ ]:
salud = cargar_procesado('Dataset_ALDIMI_Salud_Preparado.csv')
print(salud.shape)
# Constantes del proyecto (inline para autonomia en Colab)
TARGET = 'Prioridad_Atencion'
EXCLUDE_COLS = ['Patient_ID', 'Prioridad_Score', 'Score_Triage', 'Indice_Riesgo_Clinico', TARGET]  # identificadores, target y variables de leakage que no deben estar disponibles en inferencia
ORDEN = ['Bajo', 'Medio', 'Alto']
salud.head(3)

## 1.5.2 Definicion del problema y modelos evaluados

- **Random Forest Classifier**: ensamble de arboles por *bagging*; robusto, poca sensibilidad a outliers, buena interpretabilidad (feature importance).
- **XGBoost Classifier**: *gradient boosting* regularizado; suele lograr mayor exactitud, maneja bien relaciones no lineales e interacciones.

Ambos se entrenan y evaluan sobre **el mismo split** para una comparacion justa.

> **Nota de leakage**: se excluyen columnas que contienen informacion del proceso de triage o referencias internas (por ejemplo `Prioridad_Score`, `Score_Triage` e `Indice_Riesgo_Clinico`), para evitar que el modelo aprenda patrones que no estarian disponibles al momento de la prediccion.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# La cohorte ya viene filtrada (Age < 25) desde el notebook 05.
# Muestreo opcional solo si Colab sigue lento (None = usar toda la cohorte).
N_MUESTRA = None
df = salud.copy()
print(f'Cohorte pediatrico-juvenil cargada: {df.shape} | Age: {df["Age"].min()}-{df["Age"].max()}')
if N_MUESTRA and len(df) > N_MUESTRA:
    df = df.groupby(TARGET, group_keys=False).apply(
        lambda g: g.sample(int(round(N_MUESTRA * len(g) / len(salud))), random_state=42))
    print('Muestra estratificada:', df.shape)

features = [c for c in df.columns if c not in EXCLUDE_COLS]
X = df[features]
le = LabelEncoder(); y = le.fit_transform(df[TARGET].astype(str))
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
print('Numericas:', len(num_cols), '| Categoricas:', len(cat_cols))
print('Clases:', list(le.classes_))
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Pipeline: preprocesamiento + SMOTE + clasificador

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

prep = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

pipe_rf = ImbPipeline([('prep', prep), ('smote', SMOTE(random_state=42)),
                       ('clf', RandomForestClassifier(random_state=42, n_jobs=-1))])
pipe_xgb = ImbPipeline([('prep', prep), ('smote', SMOTE(random_state=42)),
                        ('clf', XGBClassifier(random_state=42, eval_metric='mlogloss', tree_method='hist'))])

## Optimizacion de hiperparametros (RandomizedSearchCV + validacion cruzada)

Se usa `RandomizedSearchCV` para optimizar hiperparametros con rigor metodologico y tiempos razonables en Colab. El SMOTE se aplica dentro del Pipeline en cada fold de entrenamiento.

### Justificacion de hiperparametros y criterios de busqueda

| Elemento | Valor | Criterio |
|---|---|---|
| **Validacion cruzada** | `StratifiedKFold(n_splits=3)` | Mantiene la proporcion de clases Bajo/Medio/Alto en cada pliegue; 3 folds equilibra sesgo-varianza y tiempo de ejecucion en Colab. |
| **Metrica de seleccion** | `f1_macro` | Promedia el F1 de las tres clases; evita optimizar solo la clase mayoritaria (Bajo). |
| **Busqueda** | `RandomizedSearchCV(n_iter=6)` | Explora el espacio de hiperparametros sin evaluar todas las combinaciones (grid completo = 64 ajustes x 2 modelos; aqui ~36 ajustes totales). |
| **`clf__n_estimators`** | `[200, 300]` | Suficientes arboles para convergencia; valores >300 aumentan tiempo en Colab sin ganancia proporcional en F1. |
| **`clf__max_depth`** | `[12, 16, None]` | Controla complejidad: profundidad limitada reduce sobreajuste; `None` permite capturar interacciones no lineales. |
| **`clf__min_samples_leaf`** | `[1, 3]` | Regularizacion: hojas con >=3 muestras estabilizan la prediccion en la clase minoritaria Alto. |
| **`clf__learning_rate`** (XGB) | `[0.05, 0.1]` | Rango estandar de boosting: 0.05 converge mas lento pero generaliza; 0.1 acelera el entrenamiento. |
| **`clf__subsample`** (XGB) | `[0.9]` | Muestreo por fila (90%) como regularizacion; valor unico por estabilidad en busqueda acotada. |
| **`n_jobs`** | `1` | En Colab, paralelizar todos los nucleos con SMOTE + OneHot suele saturar RAM; `n_jobs=1` es mas estable. |
| **SMOTE** | Dentro del Pipeline | Balanceo aplicado solo sobre el conjunto de entrenamiento de cada fold. |

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

CV_FOLDS = 3
N_ITER = 6
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)

param_dist_rf = {
    'clf__n_estimators': [200, 300],
    'clf__max_depth': [12, 16, None],
    'clf__min_samples_leaf': [1, 3],
}
param_dist_xgb = {
    'clf__n_estimators': [200, 300],
    'clf__max_depth': [4, 6],
    'clf__learning_rate': [0.05, 0.1],
    'clf__subsample': [0.9],
}

gs_rf = RandomizedSearchCV(pipe_rf, param_dist_rf, n_iter=N_ITER, scoring='f1_macro',
                           cv=cv, n_jobs=1, random_state=42, verbose=1)
gs_xgb = RandomizedSearchCV(pipe_xgb, param_dist_xgb, n_iter=N_ITER, scoring='f1_macro',
                            cv=cv, n_jobs=1, random_state=42, verbose=1)
gs_rf.fit(X_tr, y_tr)
gs_xgb.fit(X_tr, y_tr)
print('Mejor RF :', gs_rf.best_params_, '| CV f1_macro:', round(gs_rf.best_score_, 4))
print('Mejor XGB:', gs_xgb.best_params_, '| CV f1_macro:', round(gs_xgb.best_score_, 4))

## Resultados: tabla comparativa global (Tabla 1)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report
def evaluar(nombre, modelo):
    pred = modelo.predict(X_te)
    idx_alto = list(le.classes_).index('Alto')
    return {'Modelo': nombre,
            'Accuracy': round(accuracy_score(y_te, pred), 4),
            'F1_macro': round(f1_score(y_te, pred, average='macro'), 4),
            'F1_Alto': round(f1_score(y_te == idx_alto, pred == idx_alto), 4),
            'Recall_Alto': round(recall_score(y_te == idx_alto, pred == idx_alto), 4)}
tabla1 = pd.DataFrame([evaluar('Random Forest', gs_rf.best_estimator_),
                       evaluar('XGBoost', gs_xgb.best_estimator_)])
tabla1

# Cambio de leakage aplicado: se excluyen las variables de triage internamente derivadas
# para evitar que el modelo aprenda con informacion que no estaria disponible en inferencia.
# Este bloque queda justo debajo de la Tabla 1 para facilitar su inspeccion.

## Metricas por clase (Tabla 2)

In [ ]:
print('=== Random Forest ==='); print(classification_report(y_te, gs_rf.predict(X_te), target_names=le.classes_))
print('=== XGBoost ==='); print(classification_report(y_te, gs_xgb.predict(X_te), target_names=le.classes_))

> **Conclusion (seleccion del modelo):** se elige el modelo con **mayor F1_macro** y, ante empate, el de **mayor Recall en la clase Alto** (criterio critico de negocio: no dejar sin priorizar a un paciente critico). Complete la conclusion con los numeros obtenidos en Colab, indicando cual gano y por que.

## Guardado de artefactos (.joblib)

In [ ]:
import joblib
mejor_nombre = tabla1.sort_values(['F1_macro', 'Recall_Alto'], ascending=False).iloc[0]['Modelo']
mejor = gs_rf.best_estimator_ if mejor_nombre == 'Random Forest' else gs_xgb.best_estimator_
# Se guardan AMBOS modelos (para las comparativas ROC del notebook 11) y el ganador
joblib.dump(gs_rf.best_estimator_, MODELS_DIR / 'clf_random_forest.joblib')
joblib.dump(gs_xgb.best_estimator_, MODELS_DIR / 'clf_xgboost.joblib')
joblib.dump({'modelo': mejor, 'label_encoder': le, 'features': features,
             'num_cols': num_cols, 'cat_cols': cat_cols, 'nombre': mejor_nombre},
            MODELS_DIR / 'modelo_clasificacion.joblib')
tabla1.to_csv(MODELS_DIR / 'metricas_clasificacion.csv', index=False)
print('Guardado. Modelo seleccionado:', mejor_nombre)

> **Nota:** descargue la carpeta `models/` de Colab y coloquela en la raiz del proyecto para que el dashboard (`streamlit_app.py`) y el notebook 11 los usen.